## NLLB-200-Distilled-600M: Training Notebook

### Imports and Setup

In [ ]:
!pip install -q transformers datasets sentencepiece sacrebleu accelerate


In [ ]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset as TorchDataset, DataLoader
import sacrebleu
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
)
from transformers.optimization import Adafactor
import gc

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

import transformers
print("transformers version:", transformers.__version__)
print("torch version:", torch.__version__)

Using device: cuda
transformers version: 4.57.3
torch version: 2.9.0+cu126


### Model Definition

In [ ]:
MODEL_NAME = "facebook/nllb-200-distilled-600M"
MAX_LENGTH = 64
LEARNING_RATE = 3e-5
EPOCHS = 12
BATCH_SIZE = 1
GRAD_ACCUM = 16

LANG_CODES = {
    "hinglish": "hin_Deva",
    "spanglish": "spa_Latn",
    "english": "eng_Latn",
}

print("Model:", MODEL_NAME)
print("MAX_LENGTH:", MAX_LENGTH)
print("BATCH_SIZE:", BATCH_SIZE)
print("GRAD_ACCUM:", GRAD_ACCUM)
print("EPOCHS:", EPOCHS)

Model: facebook/nllb-200-distilled-600M
MAX_LENGTH: 64
BATCH_SIZE: 1
GRAD_ACCUM: 16
EPOCHS: 12


### Data Loading

In [ ]:
print("LOADING DATA")
hing_train = pd.read_csv("/content/hinglish_train.csv")
hing_val = pd.read_csv("/content/hinglish_val.csv")
hing_test = pd.read_csv("/content/hinglish_test.csv")

span_train = pd.read_csv("/content/spanglish_train.csv")
span_val = pd.read_csv("/content/spanglish_val.csv")
span_test = pd.read_csv("/content/spanglish_test.csv")

print(f"Hinglish: {len(hing_train)} train, {len(hing_val)} val, {len(hing_test)} test")
print(f"Spanglish: {len(span_train)} train, {len(span_val)} val, {len(span_test)} test")

LOADING DATA
Hinglish: 743 train, 93 val, 93 test
Spanglish: 844 train, 105 val, 106 test


### Dataset & Evaluation Utilities

In [ ]:
# This section defines all utilities needed for training and evaluating the translation model:
# - TranslationDataset: loads source–target pairs, tokenizes them, and prepares model-ready samples
# - collate_fn: dynamically pads input/label sequences for batch processing with PyTorch DataLoader
# - evaluate_model: runs generation on validation data and computes translation metrics (BLEU, chrF, EM)


class TranslationDataset(TorchDataset):
    def __init__(self, df, tokenizer, src_lang, tgt_lang, max_length):
        self.sources = df["source"].astype(str).tolist()
        self.targets = df["target"].astype(str).tolist()
        self.tokenizer = tokenizer
        self.src_lang = src_lang
        self.tgt_lang = tgt_lang
        self.max_length = max_length

    def __len__(self):
        return len(self.sources)

    def __getitem__(self, idx):
        src = self.sources[idx]
        tgt = self.targets[idx]

        self.tokenizer.src_lang = self.src_lang

        model_inputs = self.tokenizer(
            src,
            max_length=self.max_length,
            truncation=True,
            add_special_tokens=True,
        )

        labels = self.tokenizer(
            text_target=tgt,
            max_length=self.max_length,
            truncation=True,
            add_special_tokens=True,
        )

        return {
            'input_ids': model_inputs['input_ids'],
            'attention_mask': model_inputs['attention_mask'],
            'labels': labels['input_ids']
        }

def collate_fn(batch, tokenizer):
    input_ids = [item['input_ids'] for item in batch]
    attention_mask = [item['attention_mask'] for item in batch]
    labels = [item['labels'] for item in batch]

    max_input_len = max(len(ids) for ids in input_ids)
    max_label_len = max(len(lbl) for lbl in labels)

    padded_input_ids = []
    padded_attention_mask = []
    padded_labels = []

    for inp_ids, att_mask, lbl_ids in zip(input_ids, attention_mask, labels):
        input_padding = max_input_len - len(inp_ids)
        padded_input_ids.append(inp_ids + [tokenizer.pad_token_id] * input_padding)
        padded_attention_mask.append(att_mask + [0] * input_padding)

        label_padding = max_label_len - len(lbl_ids)
        padded_labels.append(lbl_ids + [-100] * label_padding)

    return {
        'input_ids': torch.tensor(padded_input_ids, dtype=torch.long),
        'attention_mask': torch.tensor(padded_attention_mask, dtype=torch.long),
        'labels': torch.tensor(padded_labels, dtype=torch.long)
    }

def evaluate_model(model, tokenizer, sources, targets, src_lang, max_length, device):
    model.eval()
    predictions = []

    tokenizer.src_lang = src_lang
    eng_token_id = tokenizer.convert_tokens_to_ids(LANG_CODES["english"])

    batch_size = 8
    for i in range(0, len(sources), batch_size):
        batch_sources = sources[i:i+batch_size]

        inputs = tokenizer(
            batch_sources,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_length
        ).to(device)

        with torch.no_grad():
            generated = model.generate(
                input_ids=inputs['input_ids'],
                attention_mask=inputs['attention_mask'],
                forced_bos_token_id=eng_token_id,
                max_length=max_length,
                num_beams=4,
                early_stopping=True,
            )

        batch_preds = tokenizer.batch_decode(generated, skip_special_tokens=True)
        predictions.extend([p.strip() for p in batch_preds])

    bleu = sacrebleu.corpus_bleu(predictions, [targets]).score
    chrf = sacrebleu.corpus_chrf(predictions, [targets]).score
    em = 100.0 * sum(p.lower() == r.lower() for p, r in zip(predictions, targets)) / len(targets)

    return predictions, bleu, chrf, em

✓ Dataset and helper functions defined


### Training the Hinglish Dataset

In [ ]:
# Hinglish Training Pipeline:
# 1. Initialize tokenizer and NLLB model for Hinglish to English generation.
# 2. Prepare the TranslationDataset and DataLoader for efficient batching.
# 3. Set up the Adafactor optimizer and enable gradient checkpointing for memory efficiency.
# 4. Train the model across multiple epochs with gradient accumulation.
# 5. After each epoch, generate translations on the validation split and compute BLEU, chrF, and EM.
# 6. Finally, save the fine-tuned model and tokenizer for later inference.

import gc
torch.cuda.empty_cache()
gc.collect()

print("\n==== HINGLISH: setting up model and data ====")

tokenizer_hing = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    src_lang=LANG_CODES["hinglish"],
    tgt_lang=LANG_CODES["english"],
)
model_hing = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model_hing.gradient_checkpointing_enable()
model_hing = model_hing.to(device)

hing_train_dataset = TranslationDataset(
    hing_train,
    tokenizer_hing,
    LANG_CODES["hinglish"],
    LANG_CODES["english"],
    MAX_LENGTH,
)

hing_train_loader = DataLoader(
    hing_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=lambda b: collate_fn(b, tokenizer_hing),
)

hing_val_sources = hing_val["source"].astype(str).tolist()
hing_val_targets = hing_val["target"].astype(str).tolist()

hing_optimizer = Adafactor(
    model_hing.parameters(),
    lr=LEARNING_RATE,
    scale_parameter=False,
    relative_step=False,
)

print("==== HINGLISH: starting training ====")
for epoch in range(EPOCHS):
    model_hing.train()
    total_loss = 0.0
    hing_optimizer.zero_grad()

    for step, batch in enumerate(hing_train_loader):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model_hing(**batch)
        loss = outputs.loss

        loss = loss / GRAD_ACCUM
        loss.backward()

        if (step + 1) % GRAD_ACCUM == 0 or (step + 1) == len(hing_train_loader):
            hing_optimizer.step()
            hing_optimizer.zero_grad()

        total_loss += loss.item()

    avg_loss = total_loss / len(hing_train_loader)
    print(f"\n[HINGLISH] Epoch {epoch+1}/{EPOCHS} - Train loss: {avg_loss:.4f}")

    _, bleu, chrf, em = evaluate_model(
        model=model_hing,
        tokenizer=tokenizer_hing,
        sources=hing_val_sources,
        targets=hing_val_targets,
        src_lang=LANG_CODES["hinglish"],
        max_length=MAX_LENGTH,
        device=device
    )

    print(f"[HINGLISH] Validation BLEU: {bleu:.2f} | chrF: {chrf:.2f} | EM: {em:.2f}%")

hing_save_dir = "/content/models/nllb_hinglish_manual"
model_hing.save_pretrained(hing_save_dir)
tokenizer_hing.save_pretrained(hing_save_dir)
print("\n✓ Hinglish model saved to:", hing_save_dir)


==== HINGLISH: setting up model and data ====
==== HINGLISH: starting training ====

[HINGLISH] Epoch 1/12 - Train loss: 0.1360
[HINGLISH] Validation BLEU: 16.31 | chrF: 41.81 | EM: 8.60%

[HINGLISH] Epoch 2/12 - Train loss: 0.1060
[HINGLISH] Validation BLEU: 19.52 | chrF: 44.21 | EM: 8.60%

[HINGLISH] Epoch 3/12 - Train loss: 0.0894
[HINGLISH] Validation BLEU: 21.16 | chrF: 46.05 | EM: 8.60%

[HINGLISH] Epoch 4/12 - Train loss: 0.0777
[HINGLISH] Validation BLEU: 24.22 | chrF: 48.54 | EM: 8.60%

[HINGLISH] Epoch 5/12 - Train loss: 0.0673
[HINGLISH] Validation BLEU: 24.15 | chrF: 48.65 | EM: 11.83%

[HINGLISH] Epoch 6/12 - Train loss: 0.0585
[HINGLISH] Validation BLEU: 24.33 | chrF: 49.19 | EM: 7.53%

[HINGLISH] Epoch 7/12 - Train loss: 0.0511
[HINGLISH] Validation BLEU: 24.57 | chrF: 49.26 | EM: 11.83%

[HINGLISH] Epoch 8/12 - Train loss: 0.0441
[HINGLISH] Validation BLEU: 26.89 | chrF: 50.05 | EM: 12.90%

[HINGLISH] Epoch 9/12 - Train loss: 0.0383
[HINGLISH] Validation BLEU: 26.07 | 

/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 200}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(



✓ Hinglish model saved to: /content/models/nllb_hinglish_manual


### Evaluating the Hinglish Dataset

In [ ]:
# Hinglish Test Evaluation:
# 1. Load test sources/targets and run batched generation with the trained model.
# 2. Decode outputs and measure translation quality using BLEU, chrF, and EM.
# 3. Store predictions in a CSV file for inspection.


print("\n==== HINGLISH: evaluating on test set ====")
model_hing.eval()

hing_test_sources = hing_test["source"].astype(str).tolist()
hing_test_targets = hing_test["target"].astype(str).tolist()
hing_test_preds = []

eng_token_id = tokenizer_hing.convert_tokens_to_ids(LANG_CODES["english"])

for i in range(0, len(hing_test_sources), 8):
    batch = hing_test_sources[i:i+8]
    tokenizer_hing.src_lang = LANG_CODES["hinglish"]
    inputs = tokenizer_hing(
        batch,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH
    ).to(device)

    with torch.no_grad():
        generated = model_hing.generate(
            **inputs,
            forced_bos_token_id=eng_token_id,
            max_length=MAX_LENGTH,
            num_beams=4
        )

    batch_preds = tokenizer_hing.batch_decode(generated, skip_special_tokens=True)
    hing_test_preds.extend([p.strip() for p in batch_preds])

hing_test_bleu = sacrebleu.corpus_bleu(hing_test_preds, [hing_test_targets]).score
hing_test_chrf = sacrebleu.corpus_chrf(hing_test_preds, [hing_test_targets]).score
hing_test_em = 100 * sum(p.lower() == r.lower() for p, r in zip(hing_test_preds, hing_test_targets)) / len(hing_test_targets)

print(f"\n[HINGLISH TEST] BLEU: {hing_test_bleu:.2f} | chrF: {hing_test_chrf:.2f} | EM: {hing_test_em:.2f}%")

hing_test["nllb_pred"] = hing_test_preds
hing_test.to_csv("/content/hinglish_nllb_predictions.csv", index=False)
print("✓ Hinglish test predictions saved")

del model_hing, tokenizer_hing
torch.cuda.empty_cache()
gc.collect()


==== HINGLISH: evaluating on test set ====

[HINGLISH TEST] BLEU: 22.19 | chrF: 49.64 | EM: 8.60%
✓ Hinglish test predictions saved


35

### Hinglish Translation Examples

In [ ]:
print("HINGLISH TRANSLATION EXAMPLES")

for i in range(7):
    print(f"\nExample {i+1}:")
    print(f"SOURCE:     {hing_test_sources[i]}")
    print(f"REFERENCE:  {hing_test_targets[i]}")
    print(f"PREDICTION: {hing_test_preds[i]}")
    print("-"*80)


HINGLISH TRANSLATION EXAMPLES

Example 1:
SOURCE:     Beach town ka mayor kuch gadbad type ka aadmi hai kyunki woh beach goers ko beach ke khatron ke baare mein nahi batana chahta.
REFERENCE:  The mayor of the beach town is kind of the bad guy as he doesn't want to tell the beach goers how dangerous the beach is.
PREDICTION: The mayor of a beach town is a very bad type of person because he doesn't want to talk about the dangers of beach goers.
--------------------------------------------------------------------------------

Example 2:
SOURCE:     Firstly, Rotten Tomatoes par ise great reviews mile ! ye ek former human ke bare me he jo apni wife ki death ka revenge lene ke liye criminal underworld me lotata he.
REFERENCE:  Firstly, it received great reviews on Rotten Tomatoes! And it's about a former human who returns to the criminal underworld to extract revenge following the death of his wife!
PREDICTION: First of all, Rotten Tomatoes has great reviews for it! It's about a former hum

### Training the Spanglish Dataset

In [ ]:
# Spanglish Training Pipeline:
# 1. Load the NLLB tokenizer/model with the correct Spanglish and English language codes.
# 2. Create the TranslationDataset for both train and validation splits.
# 3. Build DataLoaders with dynamic padding using the custom collate function.
# 4. Initialize Adafactor optimizer and enable gradient checkpointing to reduce memory usage.
# 5. Train for multiple epochs using gradient accumulation to stabilize updates.
# 6. After each epoch, run validation and compute BLEU, chrF, and exact-match metrics.
# 7. Save the fully fine-tuned Spanglish model.


torch.cuda.empty_cache()
gc.collect()

print("\n\n==== SPANGLISH: setting up model and data ====")

tokenizer_span = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    src_lang=LANG_CODES["spanglish"],
    tgt_lang=LANG_CODES["english"],
)
model_span = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model_span.gradient_checkpointing_enable()
model_span = model_span.to(device)

tokenizer_span.src_lang = LANG_CODES["spanglish"]
tokenizer_span.tgt_lang = LANG_CODES["english"]

span_train_dataset = TranslationDataset(
    span_train,
    tokenizer_span,
    LANG_CODES["spanglish"],
    LANG_CODES["english"],
    MAX_LENGTH,
)
span_val_dataset = TranslationDataset(
    span_val,
    tokenizer_span,
    LANG_CODES["spanglish"],
    LANG_CODES["english"],
    MAX_LENGTH,
)

span_train_loader = DataLoader(
    span_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=lambda b: collate_fn(b, tokenizer_span),
)
span_val_loader = DataLoader(
    span_val_dataset,
    batch_size=4,
    shuffle=False,
    collate_fn=lambda b: collate_fn(b, tokenizer_span),
)

span_val_sources = span_val["source"].astype(str).tolist()
span_val_targets = span_val["target"].astype(str).tolist()

span_optimizer = Adafactor(
    model_span.parameters(),
    lr=LEARNING_RATE,
    scale_parameter=False,
    relative_step=False,
)

print("==== SPANGLISH: starting training ====")
for epoch in range(EPOCHS):
    model_span.train()
    total_loss = 0.0
    span_optimizer.zero_grad()

    for step, batch in enumerate(span_train_loader):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model_span(**batch)
        loss = outputs.loss

        loss = loss / GRAD_ACCUM
        loss.backward()

        if (step + 1) % GRAD_ACCUM == 0 or (step + 1) == len(span_train_loader):
            span_optimizer.step()
            span_optimizer.zero_grad()

        total_loss += loss.item()

    avg_loss = total_loss / len(span_train_loader)
    print(f"\n[SPANGLISH] Epoch {epoch+1}/{EPOCHS} - Train loss: {avg_loss:.4f}")

    bleu, chrf, em = evaluate_on_val(
        model=model_span,
        tokenizer=tokenizer_span,
        val_loader=span_val_loader,
        val_sources=span_val_sources,
        val_targets=span_val_targets,
        max_length=MAX_LENGTH,
        lang_tag="spanglish",
    )
    print(f"[SPANGLISH] Validation BLEU: {bleu:.2f} | chrF: {chrf:.2f} | EM: {em:.2f}%")

span_save_dir = "/content/models/nllb_spanglish_manual"
model_span.save_pretrained(span_save_dir)
tokenizer_span.save_pretrained(span_save_dir)
print("\n✓ Spanglish model saved to:", span_save_dir)



==== SPANGLISH: setting up model and data ====
==== SPANGLISH: starting training ====

[SPANGLISH] Epoch 1/12 - Train loss: 0.0355

[SPANGLISH VALIDATION EXAMPLE]
  SOURCE:     Ese quebranto, esa apertura de mi célula de chica, esa clase de massive breakthrough de mi corazón me permitió tener más courageous, y ser más braver, y de hecho más clever de lo que fui en mi vida pasada.
  REFERENCE:  That shattering, that opening of my girl cell, that kind of massive breakthrough of my heart allowed me to become more courageous, and braver, and actually more clever than I had been in the past in my life.
  PREDICTION: That breakdown, that opening of my girl cell, that kind of massive breakthrough of my heart allowed me to be more courageous, and to be braver, and in fact, smarter than I was in my past life.
[SPANGLISH] Validation BLEU: 64.48 | chrF: 78.39 | EM: 5.71%

[SPANGLISH] Epoch 2/12 - Train loss: 0.0283

[SPANGLISH VALIDATION EXAMPLE]
  SOURCE:     Ese quebranto, esa apertura de mi 

/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 200}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(



✓ Spanglish model saved to: /content/models/nllb_spanglish_manual


### Evaluating the Spanglish Dataset

In [ ]:
print("\n==== SPANGLISH: evaluating on test set ====")
model_span.eval()

span_test_sources = span_test["source"].astype(str).tolist()
span_test_targets = span_test["target"].astype(str).tolist()
span_test_preds = []

eng_token_id_sp = tokenizer_span.convert_tokens_to_ids(LANG_CODES["english"])

for i in range(0, len(span_test_sources), 8):
    batch = span_test_sources[i:i+8]
    tokenizer_span.src_lang = LANG_CODES["spanglish"]
    inputs = tokenizer_span(
        batch,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH
    ).to(device)

    with torch.no_grad():
        generated = model_span.generate(
            **inputs,
            forced_bos_token_id=eng_token_id_sp,
            max_length=MAX_LENGTH,
            num_beams=4
        )

    batch_preds = tokenizer_span.batch_decode(generated, skip_special_tokens=True)
    span_test_preds.extend([p.strip() for p in batch_preds])

span_test_bleu = sacrebleu.corpus_bleu(span_test_preds, [span_test_targets]).score
span_test_chrf = sacrebleu.corpus_chrf(span_test_preds, [span_test_targets]).score
span_test_em = 100 * sum(p.lower() == r.lower() for p, r in zip(span_test_preds, span_test_targets)) / len(span_test_targets)

print(f"\n[SPANGLISH TEST] BLEU: {span_test_bleu:.2f} | chrF: {span_test_chrf:.2f} | EM: {span_test_em:.2f}%")

span_test["nllb_pred"] = span_test_preds
span_test.to_csv("/content/spanglish_nllb_predictions.csv", index=False)
print("✓ Spanglish test predictions saved")


==== SPANGLISH: evaluating on test set ====

[SPANGLISH TEST] BLEU: 63.12 | chrF: 77.82 | EM: 4.72%
✓ Spanglish test predictions saved


### Spanglish Translation Examples

In [ ]:
print("SPANGLISH TRANSLATION EXAMPLES")

for i in range(5):
    print(f"\nExample {i+1}:")
    print(f"SOURCE:     {span_test_sources[i]}")
    print(f"REFERENCE:  {span_test_targets[i]}")
    print(f"PREDICTION: {span_test_preds[i]}")


SPANGLISH TRANSLATION EXAMPLES

Example 1:
SOURCE:     RB: Bueno, me gusta pensar que representa quality, que, you know, if alguien se topa con una Virgin company, they -- CA: They are quality, Richard. Come on now, todo el mundo habla de quality --¿el spirit?
REFERENCE:  RB: Well, I like to think it stands for quality, that you know, if somebody comes across a Virgin company, they -- CA: They are quality, Richard. Come on now, everyone says quality. Spirit?
PREDICTION: RB: Well, I like to think it stands for quality, which, you know, if someone bumps into a Virgin company, they -- CA: They are quality, Richard. Come on now, everybody talks about quality -- the spirit?

Example 2:
SOURCE:     Presentamos más de 300 muestras de mushrooms que fueron hervidos in hot water, y el micelio harvesting estos extracellular metabolites.
REFERENCE:  We submitted over 300 samples of mushrooms that were boiled in hot water, and mycelium harvesting these extracellular metabolites.
PREDICTION: We pres